# Environment

In [ ]:
! pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 239.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 153.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 128.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
!pip install --upgrade transformers accelerate pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 19.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 39.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 242.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 208.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 185.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 29.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 70.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 203.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 153.2

## Import Library

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import json

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Model

In [2]:
! nvidia-smi

Tue Jun 24 04:44:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:85:00.0 Off |                    0 |
| N/A   28C    P0             69W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from huggingface_hub import login
import os

token = "HF_TOKEN_REDACTED"
os.environ['HF_TOKEN'] = token

login(token=token)
print("Logged in to Hugging Face!")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
Logged in to Hugging Face!


### LLama

In [4]:
# meta-llama/Llama-3.1-8B-Instruct
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.80s/it]


# Load Datasets

In [5]:
import pandas as pd
cloud=pd.read_excel("../../../data/splits/cloud/cloud_test.xlsx")
device=pd.read_excel("../../../data/splits/device/device_test.xlsx")

## Ocr Load 

In [6]:
# ocr json

import json

with open("../../../data/processed/ocr/cloud/ocr_extracted_cloud_test.jsonl", "r", encoding="utf-8") as file:
    cloud_test_ocr = [json.loads(line) for line in file]

with open("../../../data/processed/ocr/device/ocr_extracted_device_test.jsonl", "r", encoding="utf-8") as file:
    device_test_ocr = [json.loads(line) for line in file]



In [7]:
from typing import List, Dict, Any

def ocr_extract(data) -> List[str]:
    results: List[str] = []

    for entry in data:
        lines=[]

        for img_i, img in enumerate(entry.get("images", [])):
            lines.append(f"img{img_i}:")

            for resp in img.get("response", []):
                if isinstance(resp, dict):
                    # --- non-table text ---
                    ntt = resp.get("non_table_text")
                    if ntt:
                        lines.append("Non-table text:")
                        lines.extend(ntt if isinstance(ntt, list) else [str(ntt)])

                    # --- tables ---
                    tables = resp.get("table")
                    if tables:
                        lines.append("Tables:")
                        for t_i, tbl in enumerate(tables if isinstance(tables, list) else [tables]):
                            lines.append(f"Table {t_i}:")
                            lines.append(tbl.get("content", "") if isinstance(tbl, dict) else str(tbl))
                    continue

                # Unknown or malformed type
                lines.append(f"[WARN] Unhandled response type: {type(resp).__name__}")
                lines.append(str(resp))

        formatted = "\n".join(lines).strip()
        if formatted:
            results.append(formatted)

    return results

In [8]:
# format ocr response for prompt
cloud_test_ocr_response_format = ocr_extract(cloud_test_ocr)   
device_test_ocr_response_format = ocr_extract(device_test_ocr)

In [9]:
len(cloud_test_ocr_response_format),len(device_test_ocr_response_format)

(1568, 947)

In [10]:
cloud['images'][1423]

"['https://i.sstatic.net/AKSXF.png']"

In [11]:
cloud_test_ocr_response_format[1423]

'img0:\nTables:\nTable 0:\n+-----------------------------------+------------+-----------+-----------+\n| EC2 Instance Type                 | Software   | EC2       | Total     |\n+-----------------------------------+------------+-----------+-----------+\n| Standard Large (m1.large)         | $0.00/hr   | $0.265/hr | $0.265/hr |\n+-----------------------------------+------------+-----------+-----------+\n| Standard XL (m1.xlarge)           | $0.00/hr   | $0.53/hr  | $0.53/hr  |\n+-----------------------------------+------------+-----------+-----------+\n| High-Memory 2XL (m2.2xlarge)      | $0.00/hr   | $0.845/hr | $0.845/hr |\n+-----------------------------------+------------+-----------+-----------+\n| High-Memory 4XL (m2.4xlarge)      | $0.00/hr   | $1.69/hr  | $1.69/hr  |\n+-----------------------------------+------------+-----------+-----------+\n| M3 Extra Large (m3.xlarge)        | $0.00/hr   | $0.475/hr | $0.475/hr |\n+-----------------------------------+------------+-----------

## Cloud

In [12]:
cloud.head(2)

,Unnamed: 0.1,Unnamed: 0,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,0,33283,Some recently changed resources are not yet pu...,\r\n \r\nEvery once in a while ...,"['java', 'eclipse', 'google-app-engine', 'java...",https://stackoverflow.com/questions/49711724/s...,4,2018-04-07 20:34:12Z,1,"[""@code511788465541441, if you consider Chanse...",[{'body': '\n \nIf that happens...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,1,34179,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\r\n \r\nOur pipeline indicates...,"['azure-web-app-service', 'azure-deployment', ...",https://stackoverflow.com/questions/65255839/f...,7,2020-12-11 17:20:22Z,2,"[""thanks for the hint. i am going to verify th...","[{'body': ""\nI am almost certain that the pack...",\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


## Processed

In [13]:
cloud=cloud[['title','body','selected_answer','images']]

In [14]:
cloud.head(2)

,title,body,selected_answer,images
0,Some recently changed resources are not yet pu...,\r\n \r\nEvery once in a while ...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\r\n \r\nOur pipeline indicates...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


In [15]:
cloud['context'] = "title: " + cloud['title'].astype(str) + "\nbody: " + cloud['body'].astype(str).str.strip()

In [16]:
cloud_processed=pd.DataFrame(
    {
        'context':cloud['context'],
        'gt':cloud['selected_answer'],
        'img':cloud['images']
    }
)

In [17]:
cloud_processed['ocr']=cloud_test_ocr_response_format

In [18]:
cloud_processed.head(2)

,context,gt,img,ocr
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],img0:\nNon-table text:\n Stale resources dete...
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",img0:\nTables:\nTable 0:\n+----+--------------...


## Device

In [19]:
device.head(2)

,Unnamed: 0,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,0,What is the purpose of the INPUT chain in the ...,\r\n \r\nWhat is the purpose of...,"['linux', 'iptables', 'linux', 'iptables']",https://serverfault.com/questions/993878/what-...,1,2019-12-01 00:23:16Z,1,"['OK, also please add this to your answer so o...",[{'body': '\nIf some NAT operation needs to be...,\r\nIf some NAT operation needs to be only don...,['https://i.sstatic.net/NUh2k.png']
1,1,"Logstash JDBC input, filter event timestamp di...",\r\n \r\nI am trying to change ...,"['elasticsearch', 'logstash', 'elastic-stack',...",https://stackoverflow.com/questions/33787795/l...,0,2015-11-18 18:38:40Z,3,"['could you suggest how I could fix?', 'Your d...",[{'body': '\nOK I finally figure how to get th...,\r\nOK I finally figure how to get the @timest...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


## Processed

In [20]:
device=device[['title','body','selected_answer','images']]

In [21]:
device.head(2)

,title,body,selected_answer,images
0,What is the purpose of the INPUT chain in the ...,\r\n \r\nWhat is the purpose of...,\r\nIf some NAT operation needs to be only don...,['https://i.sstatic.net/NUh2k.png']
1,"Logstash JDBC input, filter event timestamp di...",\r\n \r\nI am trying to change ...,\r\nOK I finally figure how to get the @timest...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


In [22]:
device['context'] = "title: " + device['title'].astype(str) + "\nbody: " + device['body'].astype(str).str.strip()

In [23]:
device_processed=pd.DataFrame(
    {
        'context':device['context'],
        'gt':device['selected_answer'],
        'img':device['images']
    }
)

In [24]:
device_processed['ocr']=device_test_ocr_response_format

In [25]:
device_processed.head(2)

,context,gt,img,ocr
0,title: What is the purpose of the INPUT chain ...,\r\nIf some NAT operation needs to be only don...,['https://i.sstatic.net/NUh2k.png'],img0:\nNon-table text:\n CT\n ...
1,"title: Logstash JDBC input, filter event times...",\r\nOK I finally figure how to get the @timest...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",img0:\nTables:\nTable 0:\n+-------------------...


# Prompt

## Cloud

### Zero Shot

In [26]:
zero_shot = '''
You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Answer:

'''

In [27]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['ocr'],row['context'])

cloud_processed['zero-shot-question'] = cloud_processed.apply(generate_zero_shot_prompt, axis=1)


In [28]:
print(cloud_processed['zero-shot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
  Stale resources detected
                                                 ×
                                                 x
     Some recently changed resources are not yet published. Continue with launch?
 ?
                                             No
                                   Yes

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Answer:




### CoT

In [29]:
cot = '''
You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Lets think step by step:
Answer:
'''

In [30]:
def generate_cot_prompt(row):
    return cot.format(row['ocr'],row['context'])

cloud_processed['cot-question'] = cloud_processed.apply(generate_cot_prompt, axis=1)


In [31]:
print(cloud_processed['cot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
  Stale resources detected
                                                 ×
                                                 x
     Some recently changed resources are not yet published. Continue with launch?
 ?
                                             No
                                   Yes

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Lets think step by step:
Answer:



In [32]:
cloud_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1568 entries, 0 to 1567
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   context             1568 non-null   object
 1   gt                  1568 non-null   object
 2   img                 1568 non-null   object
 3   ocr                 1568 non-null   object
 4   zero-shot-question  1568 non-null   object
 5   cot-question        1568 non-null   object
dtypes: object(6)
memory usage: 73.6+ KB


## Device

### Zero Shot

In [33]:
zero_shot = '''
You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Answer:

'''

In [34]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['ocr'],row['context'])

device_processed['zero-shot-question'] =device_processed.apply(generate_zero_shot_prompt, axis=1)


In [35]:
print(device_processed['zero-shot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
       CT
                                                                                              iptables
                                                                                                                                                                              Transmit Packet to
Data Link Layer
outbound interface
              Tracked by ConnTrack
                                               Receive Packet from
Data Link Layer
inbound interface
                                                                                              raw table
PREROUTING chain
                                                                              Defrag
                                                                      

### CoT

In [36]:
cot = '''
You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Lets think step by step:
Answer:
'''

In [37]:
def generate_cot_prompt(row):
    return cot.format(row['ocr'],row['context'])

device_processed['cot-question'] = device_processed.apply(generate_cot_prompt, axis=1)


In [38]:
print(device_processed['cot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
       CT
                                                                                              iptables
                                                                                                                                                                              Transmit Packet to
Data Link Layer
outbound interface
              Tracked by ConnTrack
                                               Receive Packet from
Data Link Layer
inbound interface
                                                                                              raw table
PREROUTING chain
                                                                              Defrag
                                                                      

In [39]:
device_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947 entries, 0 to 946
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   context             947 non-null    object
 1   gt                  947 non-null    object
 2   img                 947 non-null    object
 3   ocr                 947 non-null    object
 4   zero-shot-question  947 non-null    object
 5   cot-question        947 non-null    object
dtypes: object(6)
memory usage: 44.5+ KB


## Prompt Length

In [40]:
# extract prompt length
def prompt_length(text, add_special_tokens=True):
    if pd.isnull(text) or not isinstance(text, str) or not text.strip():
        return 0
    temp_inputs = tokenizer(text, return_tensors="pt", add_special_tokens=add_special_tokens)
    return temp_inputs.input_ids.shape[1]


In [41]:
# Prompt Length -> Cloud Logs
cloud_processed['zero-shot-question-length'] = cloud_processed['zero-shot-question'].apply(prompt_length)
cloud_processed['cot-question-length'] = cloud_processed['cot-question'].apply(prompt_length)

In [42]:
# Prompt Length -> Device Logs
device_processed['zero-shot-question-length'] = device_processed['zero-shot-question'].apply(prompt_length)
device_processed['cot-question-length'] = device_processed['cot-question'].apply(prompt_length)

# Generate Response

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

### Zero-Shot

In [72]:
import os
import torch
import gc
import traceback

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.set_grad_enabled(False)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

def generate_zero_shot_answer(row):
    try:
        # Encode the question and move to the model’s device
        input_ids = tokenizer(
            row["zero-shot-question"], return_tensors="pt"
        ).input_ids.to(model.device)
        prompt_length=input_ids.shape[1]

        # Generate a completion
        output = model.generate(
            input_ids,
            max_new_tokens=768,
            repetition_penalty=1.2,
            do_sample=False,
            use_cache=True
        )

        # Decode and return the answer only
        return tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)

    except Exception as e:
        # Return the error string (no prompt info)
        return f"[ERROR] {e}\n{traceback.format_exc()}"

    finally:
        # Clean up GPU / RAM
        for var in ("input_ids", "output"):
            if var in locals():
                del locals()[var]
        torch.cuda.empty_cache()
        gc.collect()

### CoT

In [73]:
import os
import torch
import gc
import traceback

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.set_grad_enabled(False)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

def generate_cot_answer(row):
    try:
        input_ids = tokenizer(
            row["cot-question"], return_tensors="pt"
        ).input_ids.to(model.device)
        prompt_length=input_ids.shape[1]

        output = model.generate(
            input_ids,
            max_new_tokens=768,
            repetition_penalty=1.2,
            do_sample=False,
            use_cache=True,
        )

        return tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)

    except Exception as e:
        return f"[ERROR] {e}\n{traceback.format_exc()}"

    finally:
        for var in ("input_ids", "output"):
            if var in locals():
                del locals()[var]
        torch.cuda.empty_cache()
        gc.collect()


## Testing Samples

In [46]:
sample = cloud_processed.sort_values('zero-shot-question-length', ascending=False).head(3).reset_index(drop=True)

In [63]:
sample =cloud_processed[0:15]

In [64]:
sample.head(2)

,context,gt,img,ocr,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],img0:\nNon-table text:\n Stale resources dete...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,169,175
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",img0:\nTables:\nTable 0:\n+----+--------------...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,3016,3022


In [65]:
print(sample['zero-shot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
  Stale resources detected
                                                 ×
                                                 x
     Some recently changed resources are not yet published. Continue with launch?
 ?
                                             No
                                   Yes

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Answer:




In [66]:
zero_results = []
failed_responses = []
jsonl_records = []

output_file="llm_ocr_sample_llama_3_1_8b_instruct_zero_shot_results.jsonl"
failed_file="llm_ocr_sample_llama_3_1_8b_instruct_zero_shot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass

for idx, row in tqdm(sample.iterrows(), total=len(sample)):
    answer = generate_zero_shot_answer(row)

    # Track response
    zero_results.append(answer)

    # Sample print preview
    if idx == 0:
        question_text = row['zero-shot-question']
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Question:\n")
        print(short_question)

        print("\n\nSample Response:\n")
        print(answer)

    # JSONL record
    record = {
        "row": idx,
        "question": row["zero-shot-question"],
        "response": answer
    }

    jsonl_records.append(record)
    
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
             f.write(json.dumps(failed_responses) + "\n")
    

# Add generated answers to DataFrame
sample["zero-shot-generated-answer"] = zero_results

  0%|          | 0/15 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  7%|▋         | 1/15 [00:04<00:57,  4.09s/it]The attention mask and the pad token id were not set. As a c



Sample Question:


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample Response:

This is likely due to stale resources being used by your Google App Engine (GAE) server. When you make changes to your code or configuration, it may take some time for these changes to be reflected in the deployed application. Clicking 'Yes' will allow the deployment process to continue using outdated resources, resulting in running your app with old code. To resolve this issue, try checking if there are any pending updates or changes that need to be applied before redeploying your app. You can also consider increasing the timeout period for resource synchronization or adjusting your development environment settings to ensure that all changes are properly synchronized before deploying. Additionally, verifying that your Eclipse 

 13%|█▎        | 2/15 [00:14<01:43,  7.98s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 20%|██        | 3/15 [00:18<01:10,  5.86s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 27%|██▋       | 4/15 [00:21<00:53,  4.89s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 33%|███▎      | 5/15 [00:24<00:41,  4.20s/it]The attention mask and the pad token id were not set. As a c

In [71]:
print(zero_results[7])

To get the cost for each Azure Function within the Azure Function App, you can use the Azure Monitor or Azure Log Analytics tools. These services provide detailed insights into your resource usage and costs at a granular level.

In Azure Monitor, navigate to **Function Apps** > select your Function App > click on **Metrics**. Then, apply filters as needed to view specific data points such as **FunctionExecutionCount**, **FunctionExecutionUnits**, etc., grouped by individual functions.

Alternatively, using Azure Log Analytics, create a query that aggregates costs per function instance, e.g., `AzureDiagnostics | where Category == 'Functions' | summarize sum(Quantity) by Resource = functionName`.

By leveraging these tools, you'll be able to identify which Azure Functions contribute most significantly to your overall expenses, enabling informed decisions about optimizing resources and reducing unnecessary costs. No additional setup is required beyond accessing these built-in features of 

In [74]:
import json
from tqdm import tqdm

cot_results = []
failed_cot_responses = []
cot_jsonl_records = []

output_file="llm_ocr_sample_llama_3_1_8b_instruct_cot_results.jsonl"
failed_file="llm_ocr_sample_llama_3_1_8b_instruct_cot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass


for idx, row in tqdm(sample.iterrows(), total=len(sample)):
    answer = generate_cot_answer(row)

    # Append result
    cot_results.append(answer)

    # Show sample only for first entry
    if idx == 0:
        question_text = row['cot-question']
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample CoT Question:\n")
        print(short_question)

        print("\n\nSample CoT Response:\n")
        print(answer)

    # Record for JSONL
    record = {
        "index": idx,
        "question": row["cot-question"],
        "response": answer
    }
    jsonl_records.append(record)
    
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
            f.write(json.dumps(record) + "\n")
    


# Add generated CoT answers to DataFrame
sample["cot-generated-answer"] = cot_results

  0%|          | 0/15 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  7%|▋         | 1/15 [00:01<00:27,  1.94s/it]The attention mask and the pad token id were not set. As a c



Sample CoT Question:


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample CoT Response:

This is likely due to stale resources being used during deployment. When you see the prompt about some recently changed resources not being published, clicking 'Yes' will deploy your application using those stale resources instead of recompiling them. To resolve this, try invalidating the cache (cleaning) or use the `--clear_datastore` flag when deploying. This should ensure that fresh resources are used for deployment. Additionally, consider setting up automatic publishing of changes through Eclipse's settings. This way, whenever you make changes, they'll be automatically reflected without needing manual intervention.


 13%|█▎        | 2/15 [00:08<01:00,  4.62s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 20%|██        | 3/15 [00:12<00:53,  4.45s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 27%|██▋       | 4/15 [00:19<01:00,  5.54s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 33%|███▎      | 5/15 [00:24<00:53,  5.33s/it]The attention mask and the pad token id were not set. As a c

In [79]:
print(cot_results[5])

Based on your description of the issue, it seems that the problem lies within how you're interacting with the UI rather than any underlying configuration or permissions issues. You mentioned that when you right-click and inspect the HTML, changing the value for the 'Hosted' option to match the ID of the missing 'Default' queue allows you to see both options. This suggests that there might be some sort of rendering or display issue at play here.



However, without more detailed error messages or logs from either the client-side (Visual Studio) or server-side (VSTS), it's difficult to pinpoint exactly what could cause such behavior.



Given your update where installing a fresh build agent on a different environment resolves the issue, it implies that something specific to your original setup may have caused the discrepancy between what was displayed in the UI versus actual available pools.



To troubleshoot further:

* Check if there were any errors logged during the installation proc

In [ ]:
sample.to_csv("sample_llm_ocr_llama_3_1_8b_instruct.csv")

## Cloud Testset

In [80]:
cloud_testset=cloud_processed

In [81]:
cloud_testset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1568 entries, 0 to 1567
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   context                    1568 non-null   object
 1   gt                         1568 non-null   object
 2   img                        1568 non-null   object
 3   ocr                        1568 non-null   object
 4   zero-shot-question         1568 non-null   object
 5   cot-question               1568 non-null   object
 6   zero-shot-question-length  1568 non-null   int64 
 7   cot-question-length        1568 non-null   int64 
dtypes: int64(2), object(6)
memory usage: 98.1+ KB


In [82]:
cloud_testset.head(2)

,context,gt,img,ocr,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],img0:\nNon-table text:\n Stale resources dete...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,169,175
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",img0:\nTables:\nTable 0:\n+----+--------------...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,3016,3022


### zero shot response

In [83]:
import json
from tqdm import tqdm

zero_cloud_results = []
zero_cloud_jsonl_records = []
zero_cloud_failed_responses = []

output_file="llm_ocr_cloud_llama_3_1_8b_instruct_zero_shot_results.jsonl"
failed_file="llm_ocr_cloud_llama_3_1_8b_instruct_zero_shot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass

for idx, row in tqdm(cloud_testset.iterrows(), total=len(cloud_testset)):
    # Generate answer
    answer = generate_zero_shot_answer(row)
    zero_cloud_results.append(answer)

    # Show the first response as a sample
    if idx == 0:
        question_text = row['zero-shot-question']
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Cloud Zero-Shot Question:\n")
        print(short_question)

        print("\n\nSample Cloud Zero-Shot Response:\n")
        print(answer)

    # Prepare JSONL record
    record = {
        "row": idx,
        "question": row["zero-shot-question"],
        "response": answer
    }
    zero_cloud_jsonl_records.append(record)

    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
          f.write(json.dumps(record) + "\n")

# Save results in the DataFrame
cloud_testset["zero-shot-generated-answer"] = zero_cloud_results

  0%|          | 0/1568 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 1/1568 [00:04<1:47:25,  4.11s/it]The attention mask and the pad token id were not set. 



Sample Cloud Zero-Shot Question:


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample Cloud Zero-Shot Response:

This is likely due to stale resources being used by your Google App Engine (GAE) server. When you make changes to your code or configuration, it may take some time for these changes to be reflected in the deployed application. Clicking 'Yes' will allow the deployment process to continue using outdated resources, resulting in running your app with old code. To resolve this issue, try checking if there are any pending updates or changes that need to be applied before redeploying your app. You can also consider increasing the timeout period for resource synchronization or adjusting your development environment settings to ensure that all changes are properly synchronized before deploying. Additional

  0%|          | 2/1568 [00:14<3:28:40,  8.00s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 3/1568 [00:18<2:32:49,  5.86s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 4/1568 [00:21<2:07:18,  4.88s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 5/1568 [00:24<1:49:12,  4.19s/it]The attention mask and the pad token id were

In [84]:
cloud_testset.head(2)

,context,gt,img,ocr,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero-shot-generated-answer
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],img0:\nNon-table text:\n Stale resources dete...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,169,175,This is likely due to stale resources being us...
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",img0:\nTables:\nTable 0:\n+----+--------------...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,3016,3022,"Based on your description, it seems like there..."


In [85]:
cloud_testset.to_excel("../../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_llama_3_1_8b_instruct_zero_shot.xlsx", index=False)

### Cot Response

In [ ]:
import json
from tqdm import tqdm

cot_cloud_results = []
failed_cloud_cot_responses = []
cot_cloud_jsonl_records = []

output_file="llm_ocr_cloud_llama_3_1_8b_instruct_cot_results.jsonl"
failed_file="llm_ocr_cloud_llama_3_1_8b_instruct_cot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass


for idx, row in tqdm(cloud_testset.iterrows(), total=len(cloud_testset)):
    # Generate answer using CoT function
    answer = generate_cot_answer(row)
    cot_cloud_results.append(answer)

    # Sample output (first only)
    if idx == 0:
        question_text = row['cot-question']
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Cloud CoT Question:\n")
        print(short_question)

        print("\n\nSample Cloud CoT Response:\n")
        print(answer)

    # Prepare JSONL entry
    record = {
        "row": idx,
        "question": row["cot-question"],
        "response": answer
    }
    cot_cloud_jsonl_records.append(record)
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
            f.write(json.dumps(record) + "\n")

# Append column to DataFrame
cloud_testset["cot-generated-answer"] = cot_cloud_results

  0%|          | 0/1568 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 1/1568 [00:01<50:00,  1.91s/it]The attention mask and the pad token id were not set. As



Sample Cloud CoT Question:


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample Cloud CoT Response:

This is likely due to stale resources being used during deployment. When you see the prompt about some recently changed resources not being published, clicking 'Yes' will deploy your application using those stale resources instead of recompiling them. To resolve this, try invalidating the cache (cleaning) or use the `--clear_datastore` flag when deploying. This should ensure that fresh resources are used for deployment. Additionally, consider setting up automatic publishing of changes through Eclipse's settings. This way, whenever you make changes, they'll be automatically reflected without needing manual intervention.


  0%|          | 2/1568 [00:08<1:59:46,  4.59s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 3/1568 [00:12<1:55:29,  4.43s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 4/1568 [00:19<2:24:47,  5.55s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 5/1568 [00:24<2:19:17,  5.35s/it]The attention mask and the pad token id were

In [ ]:
cloud_testset.head(2)

In [ ]:
cloud_testset.to_excel("../../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_llama_3_1_8b_instruct_cot.xlsx", index=False)

## Device Testset

In [ ]:
device_testset=device_processed

In [ ]:
device_testset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1021 entries, 0 to 1020
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   context                    1021 non-null   object
 1   gt                         1021 non-null   object
 2   zero-shot-question         1021 non-null   object
 3   cot-question               1021 non-null   object
 4   zero-shot-question-length  1021 non-null   int64 
 5   cot-question-length        1021 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 48.0+ KB


In [ ]:
device_testset.shape

(1021, 6)

### Zero shot Response

In [ ]:
import json
from tqdm import tqdm

# Lists for tracking
zero_device_results = []
zero_device_jsonl_records = []
zero_device_failed_responses = []

output_file="llm_ocr_device_llama_3_1_8b_instruct_zero_shot_results.jsonl"
failed_file="llm_ocr_device_llama_3_1_8b_instruct_zero_shot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass

for idx, row in tqdm(device_testset.iterrows(), total=len(device_testset)):
    # Generate zero-shot answer
    answer = generate_zero_shot_answer(row)
    zero_device_results.append(answer)

    # Print one sample preview
    if idx == 0:
        question_text = row["zero-shot-question"]
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Device Zero-Shot Question:\n")
        print(short_question)

        print("\n\nSample Device Zero-Shot Response:\n")
        print(answer)

    # Build JSONL record
    record = {
        "row": idx,
        "question": row["zero-shot-question"],
        "response": answer
    }
    zero_device_jsonl_records.append(record)

    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
            f.write(json.dumps(record) + "\n")


# Add results to DataFrame
device_testset["zero-shot-generated-answer"] = zero_device_results

  0%|          | 1/1021 [00:32<9:10:06, 32.36s/it]



Sample Question:


You are an expert device assistant. Provide a concise answer to the following question.
If you lack sufficient information to answer, respond with "No answer."

Question:
title: What is the purpose of the INPUT chain in the


Sample Response:

There isn't enough specific context or details about how your system uses NAT rules for me to provide a precise explanation on this topic without potentially causing confusion. However, I can give some general guidance based on common practices.


In Linux's Netfilter framework, which includes iptables, there are several tables and chains used for network address translation (NAT). The main goal here seems to be to understand why one might choose to specifically target the `nat:INPUT` chain when it appears similar functionality could also be achieved through other means like the `nat:PREROUTING` chain.

1. **Understanding Chain Differences**: Each chain serves different purposes within the packet processing pipeline. Understa

100%|██████████| 1021/1021 [6:45:14<00:00, 23.81s/it] 


In [ ]:
device_testset.head(2)

,context,gt,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero-shot-generated-answer
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,144,150,There isn't enough specific context or details...
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,550,556,You're experiencing issues with Logstash's `da...


In [ ]:
device_testset.to_excel("../../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_llama_3_1_8b_instruct_zero_shot.xlsx", index=False)

### CoT Response

In [ ]:
import json
from tqdm import tqdm

# Containers
cot_device_results = []
cot_device_jsonl_records = []
cot_device_failed_responses = []

output_file="llm_ocr_device_llama_3_1_8b_instruct_cot_results.jsonl"
failed_file="llm_ocr_device_llama_3_1_8b_instruct_cot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass

for idx, row in tqdm(device_testset.iterrows(), total=len(device_testset)):
    # Generate CoT answer
    answer = generate_cot_answer(row)
    cot_device_results.append(answer)

    # Print one sample preview
    if idx == 0:
        question_text = row["cot-question"]
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Device CoT Question:\n")
        print(short_question)

        print("\n\nSample Device CoT Response:\n")
        print(answer)

    # JSONL record
    record = {
        "row": idx,
        "question": row["cot-question"],
        "response": answer
    }
    cot_device_jsonl_records.append(record)

    with open(output_file, "a") as f:
        f.write(json.dumps(jsonl_records) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
            f.write(json.dumps(failed_responses) + "\n")

# Write results back to DataFrame
device_testset["cot-generated-answer"] = cot_device_results

  0%|          | 1/1021 [00:19<5:24:59, 19.12s/it]



Sample Question:


You are an expert device assistant. Provide a concise answer to the following question.
If you lack sufficient information to answer, respond with "No answer."
Question:
title: What is the purpose of the INPUT chain in the


Sample Response:

There's no definitive rule for using one over another; however, if your network has both source NAT and destination NAT enabled on different interfaces, then it might make sense to have two separate chains - one for each type of operation. 

For example:

*   If you're performing DNAT on interface eth0 but SNAT on interface eth1, having two separate chains allows for more fine-grained control over which operations occur when packets arrive from or leave via those respective interfaces. 
*   In this scenario, packets arriving at `eth0` will undergo DNAT as they pass through the prerouting chain, while SNAT occurs after packet inspection within the routing decision process before being sent out via `eth1`. The input chain can th

100%|██████████| 1021/1021 [8:07:04<00:00, 28.62s/it] 


In [ ]:
device_testset.head(2)

,context,gt,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero-shot-generated-answer,cot-generated-answer
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,144,150,There isn't enough specific context or details...,There's no definitive rule for using one over ...
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,550,556,You're experiencing issues with Logstash's `da...,The issue lies in how Logstash interprets the ...


In [ ]:
device_testset.to_excel("../../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_llama_3_1_8b_instruct_cot.xlsx", index=False)